In [1]:
import pandas as pd
import pickle
import numpy as np

In [2]:
df = pd.read_hdf('../data/metr-la.h5')
print(f'Traffic data shape {df.shape}')
display(df.head())

with open('../data/adj_mx.pkl', 'rb') as f:
    sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(f, encoding='latin1')
print (f'Adj matrix shape: {adj_mx.shape}')

Traffic data shape (34272, 207)


,773869,767541,767542,717447,717446,717445,773062,767620,737529,717816,...,772167,769372,774204,769806,717590,717592,717595,772168,718141,769373
2012-03-01 00:00:00,64.375000,67.625000,67.125000,61.500000,66.875000,68.750000,65.125,67.125,59.625000,62.750000,...,45.625000,65.500,64.500000,66.428571,66.875,59.375000,69.000000,59.250000,69.000000,61.875
2012-03-01 00:05:00,62.666667,68.555556,65.444444,62.444444,64.444444,68.111111,65.000,65.000,57.444444,63.333333,...,50.666667,69.875,66.666667,58.555556,62.000,61.111111,64.444444,55.888889,68.444444,62.875
2012-03-01 00:10:00,64.000000,63.750000,60.000000,59.000000,66.500000,66.250000,64.500,64.250,63.875000,65.375000,...,44.125000,69.000,56.500000,59.250000,68.125,62.500000,65.625000,61.375000,69.857143,62.000
2012-03-01 00:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000
2012-03-01 00:20:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000,0.000000,0.000000,...,0.000000,0.000,0.000000,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,0.000


Adj matrix shape: (207, 207)


In [ ]:
# Data normalization

# Strips away the labels and index, just leaving the np array
traffic_data = df.values
# Gets the mean speed across all sensors and all time steps
mean_speed = np.mean(traffic_data)
# Calculates the standard deviation of the dataset
std_speed = np.std(traffic_data)
# Applies a z score allowing the neural net to work with smaller numbers
normalized_data = (traffic_data - mean_speed) / std_speed

# Generating input (X) and target (Y) sequences for training
def generate_sequences(data, history_steps=12, horizon_steps=12):
    X, Y = [], []
    # Sliding window moving across the dataset 1 step at a time
    for i in range(len(data) - history_steps - horizon_steps + 1):
        # Takes the data for 12 steps starting at i for X
        X.append(data[i : i + history_steps])
        # Takes the data for 12 steps starting at i + 12 and ending 12 steps after
        Y.append(data[i + history_steps : i + history_steps + horizon_steps])
    # Returns these as 3D matrices (total # of sequences, Time steps, number of sensors)
    return np.array(X), np.array(Y)

X_seq, Y_seq = generate_sequences(normalized_data)
print(f'Input X shape: {X_seq.shape}')
print(f'Target Y shape: {Y_seq.shape}')

Input X shape: (34249, 12, 207)
Target Y shape: (34249, 12, 207)
